# 🔗 Clase 3 — Correlación y Causalidad
### Coeficientes de correlación · Heatmaps · Análisis multivariante

---

## Contexto y objetivo de la clase

En la clase anterior aprendiste a visualizar variables de forma individual y en pares.  
Hoy vas más profundo: no solo verás **si** dos variables se relacionan, sino **cuánto**, **en qué dirección**, y —lo más importante— si esa relación implica que una **causa** a la otra.

Luego pasarás al **análisis multivariante**: qué pasa cuando tienes tres o más variables interactuando al mismo tiempo.

---

## ¿Por qué este tema importa en el mundo real?

Un analista que confunde correlación con causalidad puede llevar a un directivo a tomar decisiones completamente equivocadas.

> *"El profit baja cuando el descuento sube. ¿Debemos eliminar todos los descuentos?"*  
> Esa pregunta parece obvia. Pero la respuesta real depende de entender **por qué** están correlacionados, y si hay otras variables en juego.

---

## Dataset — Sample Superstore

El mismo que venimos usando. Variables clave para esta clase:

| Variable | Rol en esta clase |
|---|---|
| `Sales` | Variable numérica de análisis |
| `Profit` | Variable **objetivo** de análisis (la que queremos entender) |
| `Discount` | Sospechoso principal en la correlación con Profit |
| `Quantity` | Variable adicional de análisis |
| `Category`, `Region`, `Segment` | Variables para el análisis multivariante |

> ⏱️ Duración estimada: **~2 horas**

---
## 📦 Sección 1 — Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats   # librería científica: pruebas estadísticas y correlaciones

---
## 📂 Sección 2 — Carga del dataset

In [ ]:
# Carga el CSV; latin1 maneja los caracteres especiales del archivo original
df = pd.read_csv('SampleSuperstore.csv', encoding='latin1')

# Vista rápida: shape, tipos y primeras filas en una sola llamada
print(f'{df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head(3)

---
## 📐 Sección 3 — Coeficientes de correlación

### ¿Qué mide la correlación?

La correlación cuantifica la **fuerza y dirección** de la relación entre dos variables numéricas.  
El resultado es siempre un número entre −1 y +1:

```
 r = +1.0  →  relación positiva perfecta   (cuando A sube, B siempre sube igual)
 r =  0.0  →  sin relación lineal
 r = -1.0  →  relación negativa perfecta   (cuando A sube, B siempre baja igual)
```

### Guía de interpretación práctica

| Rango de |r| | Interpretación |
|---|---|
| 0.00 – 0.19 | Correlación despreciable |
| 0.20 – 0.39 | Correlación débil |
| 0.40 – 0.59 | Correlación moderada |
| 0.60 – 0.79 | Correlación fuerte |
| 0.80 – 1.00 | Correlación muy fuerte |

### Tipos de coeficiente

Existen dos coeficientes principales y cada uno responde a una pregunta diferente:

| Coeficiente | Mide | Cuándo usarlo |
|---|---|---|
| **Pearson (r)** | Relación **lineal** entre variables continuas | Variables numéricas sin muchos outliers |
| **Spearman (ρ)** | Relación **monótona** (por rangos) | Outliers fuertes, variables ordinales o relaciones no lineales |

> **Pearson** asume que la relación es una línea recta.  
> **Spearman** solo asume que cuando A sube, B tiende a subir (o bajar) — sin importar la forma exacta.

### 3.1 — Tabla de correlación: el punto de partida

Antes de graficar, siempre calcula la tabla numérica. Te da el valor exacto de cada par.

In [ ]:
# Selecciona las 4 variables numéricas de negocio (excluye Postal Code que es un ID)
numericas = ['Sales', 'Quantity', 'Discount', 'Profit']

# .corr() calcula el coeficiente de Pearson entre todos los pares de columnas seleccionadas
# El resultado es una matriz simétrica: corr[A][B] == corr[B][A]
# La diagonal siempre es 1.0 (cada variable se correlaciona perfectamente consigo misma)
tabla_pearson = df[numericas].corr(method='pearson')

tabla_pearson.round(3)

### 3.2 — Heatmap de Pearson

La tabla de números es precisa pero difícil de leer de un vistazo.  
El heatmap la convierte en un mapa de calor donde el color comunica la fuerza y dirección instantáneamente.

**Cómo leerlo:**
- 🔴 Rojo intenso → correlación positiva fuerte
- ⚪ Blanco → sin correlación (r ≈ 0)
- 🔵 Azul intenso → correlación negativa fuerte

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

sns.heatmap(
    tabla_pearson,
    annot=True,          # escribe el coeficiente numérico dentro de cada celda
    fmt='.2f',           # dos decimales para que no ocupe demasiado espacio
    cmap='coolwarm',     # rojo = positivo · blanco = 0 · azul = negativo
    center=0,            # ancla el blanco del colormap exactamente en r = 0
    vmin=-1, vmax=1,     # fuerza la escala de color de -1 a +1 siempre (no se adapta a los datos)
    square=True,         # celdas cuadradas: más fácil de leer
    linewidths=0.5,      # líneas divisorias entre celdas
    ax=ax
)
ax.set_title('Correlación de Pearson — Variables numéricas')
plt.tight_layout()
plt.show()

### 3.3 — Correlación de Spearman

Pearson tiene una debilidad: es sensible a **outliers** y solo detecta relaciones **lineales**.

Spearman trabaja con los **rangos** de los datos, no con los valores originales.  
En lugar de preguntar "¿cuánto cambia B cuando A sube X unidades?", pregunta "¿cuando A ocupa el puesto N en su ranking, B también tiende a ocupar un puesto alto en el suyo?".

**¿Por qué importa para este dataset?**  
`Sales` tiene outliers enormes (máx $22 638 con media $229). Esos extremos pueden inflar o distorsionar el coeficiente de Pearson. Spearman los neutraliza porque trabaja con el orden, no con el valor absoluto.

In [ ]:
# method='spearman' ordena los datos por rango antes de calcular la correlación
# Esto elimina el efecto de los outliers extremos
tabla_spearman = df[numericas].corr(method='spearman')

tabla_spearman.round(3)

### 3.4 — Comparación Pearson vs Spearman

Cuando los dos coeficientes difieren mucho para el mismo par de variables, es una señal de que:
1. Hay **outliers** que están distorsionando Pearson, o
2. La relación es **no lineal** (Pearson no la captura bien)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Heatmap Pearson ────────────────────────────────────────────────────────
sns.heatmap(tabla_pearson, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=axes[0])
axes[0].set_title('Pearson (lineal · sensible a outliers)')

# ── Heatmap Spearman ───────────────────────────────────────────────────────
sns.heatmap(tabla_spearman, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=axes[1])
axes[1].set_title('Spearman (rangos · robusto a outliers)')

plt.suptitle('Pearson vs Spearman — ¿cambian mucho los valores?', y=1.02)
plt.tight_layout()
plt.show()

### 3.5 — Correlación de cada variable con el target (Profit)

En proyectos reales, el primer paso es identificar qué variables tienen mayor correlación con la variable que quieres entender o predecir.  
Este análisis guía todo lo que viene después: qué cruzar, qué modelar, qué investigar más.

In [ ]:
# Extrae solo la fila/columna de Profit de la matriz de correlación
# .drop('Profit') evita mostrar la correlación de Profit consigo mismo (siempre 1.0)
corr_con_profit = tabla_pearson['Profit'].drop('Profit').sort_values()

fig, ax = plt.subplots(figsize=(6, 3))

# Color condicional: rojo para correlación negativa, verde para positiva
colores = ['#d73027' if v < 0 else '#4dac26' for v in corr_con_profit]
corr_con_profit.plot(kind='barh', ax=ax, color=colores, edgecolor='white')

ax.axvline(0, color='black', linewidth=0.8)   # línea de referencia en r = 0
ax.set_title('Correlación de Pearson de cada variable con Profit')
ax.set_xlabel('Coeficiente r')
plt.tight_layout()
plt.show()

---
## ⚠️ Sección 4 — Correlación ≠ Causalidad

Este es uno de los conceptos más importantes —y más malinterpretados— en análisis de datos.

### ¿Qué significa que dos variables estén correlacionadas?

Solo significa que **tienden a cambiar juntas**. No dice nada sobre por qué.

Hay tres explicaciones posibles para una correlación:

```
Caso 1: A causa B
        Descuento ──causa──▶ Pérdida de profit

Caso 2: B causa A
        La pérdida de profit ──lleva a──▶ aplicar más descuentos para mover inventario

Caso 3: C causa tanto A como B (variable confusora)
        Categoría poco rentable ──▶ necesita descuentos ──▶ y también genera pérdidas
        (el descuento y la pérdida suben juntos, pero no se causan entre sí)
```

### Ejemplos reales de correlaciones sin causalidad

- Las ventas de helado correlacionan con los ahogamientos en piscinas. ¿El helado mata? No: el calor del verano causa ambas.
- Los países con más televisores per cápita tienen mayor esperanza de vida. ¿Ver TV alarga la vida? No: la riqueza causa ambas.

> **Regla de oro:** Para establecer causalidad necesitas un **experimento controlado** o técnicas econométricas específicas. La correlación sola nunca es suficiente.

In [ ]:
# Visualiza la correlación Discount vs Profit — parece clara y directa
fig, ax = plt.subplots(figsize=(7, 4))

ax.scatter(
    df['Discount'], df['Profit'],
    alpha=0.2,       # transparencia para ver densidad con 9994 puntos
    s=15,            # tamaño de cada punto
    color='steelblue'
)
ax.axhline(0, color='red', linestyle='--', linewidth=1.5, label='Profit = $0')
ax.set_title('Discount vs Profit — correlación visible, ¿pero es causal?')
ax.set_xlabel('Descuento aplicado')
ax.set_ylabel('Profit ($)')
ax.legend()
plt.tight_layout()
plt.show()

### 4.1 — Investigando la variable confusora: ¿el descuento causa pérdida o hay algo más?

Si el descuento *causara* la pérdida directamente, esperaríamos verlo en **todas** las categorías por igual.  
Pero si alguna categoría pierde dinero incluso **sin descuento**, entonces hay algo más estructural en juego.

In [ ]:
# Agrupa por Category y Discount para ver el profit promedio en cada combinación
# Esto nos permite ver si el efecto del descuento es igual en todas las categorías
pivot_cat = df.pivot_table(
    values='Profit',
    index='Discount',        # nivel de descuento en el eje Y
    columns='Category',      # una columna por categoría
    aggfunc='mean'           # profit promedio en cada celda
)

fig, ax = plt.subplots(figsize=(8, 5))

# Una línea por categoría: si las líneas caen de forma diferente, el efecto no es homogéneo
for cat in pivot_cat.columns:
    ax.plot(pivot_cat.index, pivot_cat[cat], marker='o', markersize=4, label=cat)

ax.axhline(0, color='red', linestyle='--', linewidth=1, label='Breakeven $0')
ax.set_title('Profit promedio por nivel de descuento — separado por Categoría')
ax.set_xlabel('Descuento aplicado')
ax.set_ylabel('Profit promedio ($)')
ax.legend(title='Categoría')
plt.tight_layout()
plt.show()

---
## 🔀 Sección 5 — Análisis Multivariante

### ¿Por qué necesitamos más de dos variables?

El mundo real es multivariante. Aislar dos variables es útil para empezar, pero raramente la historia completa está en un solo par.

El análisis multivariante busca entender **cómo interactúan tres o más variables al mismo tiempo** — algo que los análisis bivariados no pueden revelar.

**Ejemplo de interacción:**  
Descuento correlaciona negativamente con Profit. Pero ¿es igual en todas las regiones? ¿En todos los segmentos de cliente? ¿En todas las categorías?  
Si no es igual, hay una **interacción** entre variables: el efecto de una depende del nivel de otra.

### Herramientas para análisis multivariante

| Herramienta | Variables que maneja | Úsala cuando... |
|---|---|---|
| **Pairplot** | Todas las numéricas | Quieres una exploración rápida de todos los pares |
| **Scatter con hue** | 2 numéricas + 1 categórica | Quieres ver si una tercera variable cambia el patrón |
| **regplot / lmplot** | 2 numéricas + grupos | Quieres ver la tendencia lineal con intervalo de confianza |
| **FacetGrid / catplot** | Cualquier combinación | Quieres repetir la misma gráfica para cada subgrupo |
| **Heatmap de pivot** | 2 categóricas + 1 numérica | Quieres ver una métrica en todas las combinaciones posibles |

### 5.1 — Pairplot: exploración rápida de todos los pares

El pairplot es la herramienta de exploración multivariante más eficiente.  
Con **una sola línea** genera una matriz de gráficas donde:
- La **diagonal** muestra la distribución de cada variable (KDE o histograma)
- Los **triángulos** superior e inferior muestran el scatter entre cada par

Con el parámetro `hue` añades una variable categórica como color, lo que permite ver si los grupos se comportan diferente.

In [ ]:
# Toma una muestra de 1500 filas para que el pairplot sea rápido de renderizar
# Con 9994 puntos sería muy lento y difícil de leer
muestra = df[['Sales', 'Quantity', 'Discount', 'Profit', 'Category']].sample(1500, random_state=42)

# pairplot genera una matriz de scatter para todos los pares de variables numéricas
# hue='Category' colorea cada punto según la categoría → revela si los grupos siguen patrones distintos
# diag_kind='kde' pone una curva KDE en la diagonal (más informativa que un histograma simple)
# plot_kws ajusta la transparencia de los puntos para manejar el solapamiento
sns.pairplot(
    muestra,
    hue='Category',
    diag_kind='kde',
    plot_kws={'alpha': 0.4, 's': 15}
)
plt.suptitle('Pairplot — todos los pares de variables numéricas por Categoría', y=1.01)
plt.show()

### 5.2 — Scatter con tercera variable (hue)

El scatter básico muestra la relación entre dos variables numéricas.  
Agregar `hue` introduce una **tercera dimensión** mediante color: ¿el patrón es igual en todos los grupos?

Si los grupos forman **nubes separadas** o tienen **pendientes diferentes**, hay una interacción que el análisis bivariado nunca hubiera revelado.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.scatterplot(
    data=df,
    x='Discount', y='Profit',
    hue='Category',         # tercera variable: color diferente por categoría
    alpha=0.35,             # transparencia para ver densidad con miles de puntos
    s=18,                   # tamaño de cada punto
    ax=ax
)
ax.axhline(0, color='red', linestyle='--', linewidth=1.2, label='Breakeven $0')
ax.set_title('Discount vs Profit — ¿el patrón cambia según la Categoría?')
ax.set_xlabel('Descuento aplicado')
ax.set_ylabel('Profit ($)')
plt.tight_layout()
plt.show()

### 5.3 — Regplot: línea de tendencia con intervalo de confianza

`regplot` dibuja el scatter más una **línea de regresión lineal** ajustada a los datos,  
junto con una **banda de confianza** (área sombreada) que indica la incertidumbre del ajuste.

**¿Qué es la banda de confianza?**  
Es el rango donde, con 95% de probabilidad, cae la verdadera línea de relación entre las dos variables.  
Una banda ancha = poca certeza. Una banda estrecha = la relación es muy consistente.

**¿Para qué sirve en análisis de datos?**  
Permite ver visualmente si la tendencia lineal es una buena aproximación o si los datos se dispersan demasiado alrededor de ella.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

# regplot combina scatter + línea de regresión OLS + banda de confianza del 95%
# ci=95 es el valor por defecto del intervalo de confianza (se puede cambiar)
# scatter_kws y line_kws controlan el estilo del scatter y la línea respectivamente
sns.regplot(
    data=df,
    x='Discount', y='Profit',
    scatter_kws={'alpha': 0.15, 's': 12, 'color': 'steelblue'},  # puntos semitransparentes
    line_kws={'color': 'crimson', 'linewidth': 2},                # línea de tendencia en rojo
    ci=95          # banda de confianza del 95%
)
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.title('Regresión lineal: Discount → Profit (con intervalo de confianza 95%)')
plt.xlabel('Descuento aplicado')
plt.ylabel('Profit ($)')
plt.tight_layout()
plt.show()

### 5.4 — lmplot: regresión separada por grupos

`lmplot` hace lo mismo que `regplot` pero permite separar la regresión por grupos usando `hue` o `col`.

**La diferencia con regplot:**  
- `regplot` → un solo scatter + una sola línea
- `lmplot` → un scatter + una línea **por cada categoría**, con sus propias bandas de confianza

Si las líneas tienen **pendientes muy distintas**, eso confirma que existe una **interacción** entre el descuento y la categoría sobre el profit. El efecto del descuento no es el mismo para todos los productos.

In [ ]:
# lmplot genera su propia figura internamente (no se puede pasar ax=)
# col='Category' crea un panel separado para cada categoría
# height y aspect controlan el tamaño de cada panel individual
g = sns.lmplot(
    data=df,
    x='Discount', y='Profit',
    col='Category',          # un panel por categoría: fácil comparar pendientes
    scatter_kws={'alpha': 0.2, 's': 10},
    line_kws={'color': 'crimson'},
    height=4, aspect=0.9,
    ci=95
)
g.set_axis_labels('Descuento', 'Profit ($)')
g.figure.suptitle('Regresión Discount → Profit por Categoría — ¿la pendiente cambia?', y=1.03)
plt.show()

### 5.5 — Catplot: distribución de una métrica en múltiples subgrupos

`catplot` es la versión "facetada" de los gráficos categóricos (boxplot, barplot, violinplot).  
Repite la misma gráfica para cada subgrupo usando `col` o `row`, lo que permite comparar distribuciones de forma sistemática.

**¿Cuándo es mejor que un barplot agrupado?**  
Cuando tienes muchas combinaciones de variables. Un barplot agrupado con 4 regiones × 3 categorías × 3 segmentos sería ilegible. Con `catplot` y `col=` cada subgrupo tiene su propio espacio.

In [ ]:
# kind='box' genera boxplots; otras opciones: 'bar', 'violin', 'strip', 'swarm'
# col='Region' crea un panel por región para ver si el patrón regional cambia
g = sns.catplot(
    data=df,
    x='Category', y='Profit',
    col='Region',           # un panel por región
    kind='box',             # tipo de gráfica categórica
    height=4, aspect=0.75,
    palette='Set2',
    width=0.5
)
g.map(plt.axhline, y=0, color='red', linestyle='--', linewidth=1)  # añade línea de $0 a cada panel
g.set_axis_labels('Categoría', 'Profit ($)')
g.set_titles(col_template='Región: {col_name}')                    # título de cada panel
g.figure.suptitle('Distribución de Profit por Categoría y Región', y=1.03)
plt.show()

### 5.6 — Heatmap multivariante: interacción de dos categóricas sobre una numérica

El heatmap de tabla pivote es la herramienta más directa para visualizar **cómo interactúan dos variables categóricas** sobre una métrica numérica.

Cada celda responde a la pregunta: *"¿Cuánto profit genera la combinación Sub-Categoría X en la Región Y?"*  
Lo que buscamos son **celdas rojas en filas o columnas específicas** — eso indica que el problema no es generalizado sino que está localizado en una combinación concreta.

In [ ]:
# Tabla pivote: Sub-Category en filas, Region en columnas, Profit total en cada celda
pivot = df.pivot_table(
    values='Profit',
    index='Sub-Category',    # 17 sub-categorías en las filas
    columns='Region',        # 4 regiones en las columnas
    aggfunc='sum'            # suma acumulada de profit (no promedio)
).round(0)

# Ordena las filas de menor a mayor profit total para que las problemáticas queden arriba
pivot = pivot.loc[pivot.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(8, 8))

sns.heatmap(
    pivot,
    annot=True,              # valor numérico dentro de cada celda
    fmt=',.0f',              # miles separados, sin decimales
    cmap='RdYlGn',           # rojo = pérdida · amarillo = neutro · verde = ganancia
    center=0,                # el amarillo se ancla en profit = 0
    linewidths=0.4,
    ax=ax
)
ax.set_title('Profit Total ($) — Sub-Categoría × Región
(rojo = pierde dinero en esa combinación)')
ax.set_ylabel('')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

---
## 🧪 Sección 6 — ¿La correlación es estadísticamente significativa?

Hasta ahora calculamos coeficientes de correlación. Pero un coeficiente de r = −0.30 puede ser real o puede ser producto del azar si la muestra es pequeña.

La **prueba de significancia estadística** responde: *¿qué tan probable es observar este coeficiente si en realidad no hubiera relación?*

El resultado es el **p-valor**:
- `p < 0.05` → la correlación es estadísticamente significativa (poco probable que sea azar)
- `p ≥ 0.05` → no tenemos suficiente evidencia para descartar el azar

> ⚠️ Significancia estadística ≠ importancia práctica.  
> Con muestras grandes (como nuestras 9 994 filas) casi cualquier correlación resulta significativa, aunque el efecto sea pequeño.

In [ ]:
# scipy.stats.pearsonr calcula el coeficiente r Y el p-valor en una sola llamada
# Devuelve una tupla: (coeficiente, p_valor)
pares = [('Discount', 'Profit'), ('Sales', 'Profit'), ('Quantity', 'Profit')]

print(f"{'Par de variables':<30} {'r de Pearson':>15} {'p-valor':>15} {'Significativa':>15}")
print('-' * 75)

for var_a, var_b in pares:
    # Elimina filas con NaN en cualquiera de las dos variables antes de calcular
    datos_limpios = df[[var_a, var_b]].dropna()
    resultado = stats.pearsonr(datos_limpios[var_a], datos_limpios[var_b])  # devuelve (r, p)
    r_val  = resultado.statistic   # coeficiente de Pearson
    p_val  = resultado.pvalue      # p-valor de la prueba de hipótesis
    sig    = '✅ Sí' if p_val < 0.05 else '❌ No'
    print(f"{f'{var_a} vs {var_b}':<30} {r_val:>15.4f} {p_val:>15.2e} {sig:>15}")

---
## 🏁 Resumen de la clase

### Herramientas aprendidas

| Herramienta | Función | Para qué sirve |
|---|---|---|
| `.corr(method='pearson')` | pandas | Matriz de correlación lineal |
| `.corr(method='spearman')` | pandas | Matriz de correlación por rangos (robusta a outliers) |
| `sns.heatmap()` | seaborn | Visualización de matrices de correlación |
| `sns.pairplot()` | seaborn | Exploración rápida de todos los pares numéricos |
| `sns.regplot()` | seaborn | Scatter + línea de regresión + banda de confianza |
| `sns.lmplot()` | seaborn | Regresión separada por grupos |
| `sns.catplot()` | seaborn | Gráficas categóricas repetidas por subgrupos |
| `df.pivot_table()` + heatmap | pandas + seaborn | Interacción de dos categóricas sobre una numérica |
| `stats.pearsonr()` | scipy | Coeficiente r + p-valor de significancia |

---

### Los 3 conceptos más importantes de hoy

> **1. Pearson vs Spearman:** Usa Spearman cuando haya outliers o relaciones no lineales.  
> **2. Correlación ≠ Causalidad:** Una correlación puede explicarse por una tercera variable (confusora). Siempre pregunta *¿por qué?*  
> **3. Las interacciones cambian todo:** El efecto del descuento no es igual en Furniture que en Technology. El análisis multivariante reveló lo que el bivariado ocultaba.

---

### 🧠 Preguntas para reflexionar

1. Pearson y Spearman dieron valores distintos para el mismo par de variables. ¿Cuál deberías reportar en este dataset y por qué?
2. El heatmap Sub-Categoría × Región tiene celdas rojas en filas específicas pero no en columnas completas. ¿Qué le dirías al directivo de esa región?
3. En `lmplot`, las pendientes de las tres categorías son distintas. ¿Cómo explicarías eso a alguien sin formación estadística?
4. El p-valor de una correlación fue < 0.05. ¿Eso significa que el descuento *causa* la pérdida de profit?